# NB08：ReAct 與 Tool Calling — 讓 LLM 主動使用工具

**目標：** 理解 ReAct（Reasoning + Acting）框架與 OpenAI Tool Calling API，從零實作一個能夠主動呼叫工具的 LLM Agent。

**學習成果：**
- 理解 OpenAI `tool_calls` 格式與函數呼叫 API
- 實作工具注冊表（Tool Registry），包含 4 個工具：`web_search`、`calculator`、`rag_search`、`get_current_time`
- 從零實作 ReAct 迴圈：Thought → Action → Observation → 重複直到 Final Answer
- 加入自我反思（Self-Reflection）步驟，在回傳最終答案前進行批判性檢查
- 加入停止條件（`max_iterations=6`）與錯誤處理

**系列位置：** NB01（基礎）→ NB02（Chunking）→ NB03（混合搜索）→ NB04（評估）→ NB05（Query轉換）→ NB06（進階索引）→ NB07（生產優化）→ **NB08（ReAct + Tool Calling）**

---
## Part 0：環境設定

載入必要套件、讀取 `.env` 環境變數、初始化 OpenAI 客戶端與 ChromaDB。

In [ ]:
# pip install openai chromadb python-dotenv (if not already installed)
import os
import json
import math
import datetime
import re
from typing import Any, Callable, Dict, List, Optional

from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

print("✓ OpenAI client 初始化完成")
print(f"  Chat model  : {CHAT_MODEL}")
print(f"  Embed model : {EMBED_MODEL}")

In [ ]:
# ── ChromaDB（in-memory）設定 ──────────────────────────────────────
chroma_client = chromadb.Client()  # in-memory，無需磁碟

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBED_MODEL,
)

collection = chroma_client.get_or_create_collection(
    name="nb08_knowledge_base",
    embedding_function=openai_ef,
)

print("✓ ChromaDB in-memory collection 建立完成")

In [ ]:
# ── 模擬知識庫：10 篇關於 LLM / RAG / 雲端主題的中文文件 ──────────
KB_DOCS = [
    {
        "id": "doc_001",
        "text": "RAG（Retrieval-Augmented Generation）是一種將資訊檢索與語言模型生成結合的架構。"
                "它先從向量資料庫中找到相關文件，再將這些文件作為上下文提供給 LLM，"
                "大幅減少幻覺（hallucination）並提升回答的事實正確性。",
        "topic": "RAG",
    },
    {
        "id": "doc_002",
        "text": "向量嵌入（Vector Embedding）是將文字轉換成高維度數值向量的技術。"
                "OpenAI 的 text-embedding-3-small 模型可產生 1536 維向量，"
                "透過餘弦相似度（cosine similarity）計算語意相近程度。",
        "topic": "Embedding",
    },
    {
        "id": "doc_003",
        "text": "ReAct（Reasoning and Acting）框架由 Yao et al. 2022 提出，"
                "讓 LLM 在思考（Thought）與行動（Action）之間交替進行，"
                "並觀察（Observation）每次行動的結果，藉此完成複雜的多步驟推理任務。",
        "topic": "ReAct",
    },
    {
        "id": "doc_004",
        "text": "Tool Calling（工具呼叫）讓 LLM 能夠以結構化 JSON 格式描述它想呼叫的函數，"
                "包含函數名稱與參數。開發者解析這個 JSON 後執行對應函數，"
                "再把結果透過 tool 角色的訊息回傳給模型，形成完整的 Agentic 迴圈。",
        "topic": "ToolCalling",
    },
    {
        "id": "doc_005",
        "text": "Google Cloud Vertex AI 提供了托管式的向量搜索服務（Vector Search），"
                "支援十億級別的向量索引，適合大規模 RAG 應用。"
                "它與 BigQuery、Dataflow 深度整合，方便企業建立端到端的 AI Pipeline。",
        "topic": "Cloud",
    },
    {
        "id": "doc_006",
        "text": "AWS Bedrock 是亞馬遜的全受管基礎模型服務，支援 Anthropic Claude、"
                "Meta Llama、Mistral 等多家廠商的模型。"
                "開發者無需管理基礎設施即可呼叫最新的 LLM，並整合 Knowledge Bases 功能實現 RAG。",
        "topic": "Cloud",
    },
    {
        "id": "doc_007",
        "text": "Prompt Engineering 的核心技巧包含：Chain-of-Thought（思維鏈）、"
                "Few-Shot（少量示例）、Role Prompting（角色扮演）。"
                "其中 CoT 對需要多步計算或推理的任務效果最為顯著。",
        "topic": "PromptEngineering",
    },
    {
        "id": "doc_008",
        "text": "Self-RAG 是一種讓模型學會判斷「是否需要檢索」的技術，"
                "透過訓練模型輸出特殊的反思 token（如 [Retrieve]、[ISREL]），"
                "避免不必要的檢索浪費，同時提升回答品質。",
        "topic": "RAG",
    },
    {
        "id": "doc_009",
        "text": "ChromaDB 是一個開源的向量資料庫，支援記憶體模式（in-memory）與持久化模式。"
                "它提供 Python / JavaScript SDK，並內建多種距離函數（L2、cosine、IP），"
                "非常適合快速原型開發與學習用途。",
        "topic": "VectorDB",
    },
    {
        "id": "doc_010",
        "text": "Agent（智能代理）與傳統 LLM 應用的關鍵差異在於「主動性」："
                "Agent 能夠根據任務動態決定要採取哪些步驟、呼叫哪些工具，"
                "而非只是被動地依照固定流程回應。典型的 Agent 架構包含："
                "規劃器（Planner）、記憶體（Memory）、工具集（Tools）與執行器（Executor）。",
        "topic": "Agent",
    },
]

# 將文件寫入 ChromaDB
collection.upsert(
    ids=[d["id"] for d in KB_DOCS],
    documents=[d["text"] for d in KB_DOCS],
    metadatas=[{"topic": d["topic"]} for d in KB_DOCS],
)

print(f"✓ 已將 {len(KB_DOCS)} 篇文件寫入 ChromaDB")

---
## Part 1：Tool 定義與工具注冊表

### 1-1 四個工具的 Python 實作

我們實作四個工具：

| 工具名稱 | 功能 | 備註 |
|----------|------|------|
| `web_search` | 模擬網路搜索，依關鍵字回傳假新聞片段 | Mock 實作 |
| `calculator` | 計算數學表達式 | 使用 Python `eval`（安全版本）|
| `rag_search` | 向量語意搜索知識庫 | 使用 ChromaDB |
| `get_current_time` | 取得當前日期時間 | 系統時間 |

In [ ]:
# ── Tool 1：模擬網路搜索 ──────────────────────────────────────────
MOCK_WEB_DB = {
    "gpt": "[網路片段] OpenAI 於 2024 年底發布 GPT-4o，支援多模態輸入，在多項基準測試中超越前代模型。",
    "gemini": "[網路片段] Google DeepMind 的 Gemini 1.5 Pro 支援最長 100 萬 token 的上下文視窗，創下業界紀錄。",
    "claude": "[網路片段] Anthropic Claude 3.5 Sonnet 在編程與推理任務上達到業界頂尖水準，並支援 Computer Use 功能。",
    "rag": "[網路片段] 2024 年 RAG 技術持續演進，GraphRAG（微軟）與 HyDE 等方法顯著提升了複雜問答的準確率。",
    "agent": "[網路片段] AI Agent 在 2025 年進入爆發期，各大雲端廠商紛紛推出 Agent 編排服務，如 AWS Bedrock Agents。",
    "python": "[網路片段] Python 3.12 帶來顯著效能提升，預計 3.13 版本的 GIL 移除特性將大幅改善多線程性能。",
    "vector": "[網路片段] 向量資料庫市場持續擴大，Pinecone、Weaviate、Qdrant 相繼完成大型融資輪。",
}

def web_search(query: str) -> str:
    """模擬網路搜索，依關鍵字匹配回傳相關片段。"""
    query_lower = query.lower()
    results = []
    for keyword, snippet in MOCK_WEB_DB.items():
        if keyword in query_lower:
            results.append(snippet)
    if results:
        return "\n".join(results)
    return f"[網路搜索] 找不到與「{query}」高度相關的結果，請嘗試其他關鍵字。"


# ── Tool 2：計算機 ────────────────────────────────────────────────
def calculator(expression: str) -> str:
    """安全地計算數學表達式，支援加減乘除、次方、sqrt 等。"""
    # 白名單：只允許數字、運算符號、空白、括號與常用數學函數
    allowed = re.compile(r'^[0-9\s\+\-\*/\.\(\)\^\%]+$')
    # 也允許 math 函數調用
    safe_expr = expression.replace("^", "**")  # 支援 ^ 作為次方
    try:
        # 在受限命名空間中執行
        safe_globals = {"__builtins__": {}, "sqrt": math.sqrt,
                        "sin": math.sin, "cos": math.cos,
                        "log": math.log, "pi": math.pi, "e": math.e}
        result = eval(safe_expr, safe_globals)  # noqa: S307
        return f"計算結果：{expression} = {result}"
    except Exception as exc:
        return f"計算錯誤：{exc}"


# ── Tool 3：RAG 語意搜索 ──────────────────────────────────────────
def rag_search(query: str, n_results: int = 3) -> str:
    """從 ChromaDB 知識庫中找到最相關的文件片段。"""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
    )
    docs = results["documents"][0]
    if not docs:
        return "知識庫中找不到相關資訊。"
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"[知識庫文件 {i}]\n{doc}")
    return "\n\n".join(formatted)


# ── Tool 4：取得當前時間 ──────────────────────────────────────────
def get_current_time() -> str:
    """回傳當前的日期與時間（UTC+8 台灣時間）。"""
    now = datetime.datetime.now()
    return f"當前時間：{now.strftime('%Y-%m-%d %H:%M:%S')}（本地時間）"


# ── 工具注冊表（Tool Registry）────────────────────────────────────
TOOL_REGISTRY: Dict[str, Callable] = {
    "web_search": web_search,
    "calculator": calculator,
    "rag_search": rag_search,
    "get_current_time": get_current_time,
}

print("✓ 工具注冊表建立完成")
print("  已注冊工具:", list(TOOL_REGISTRY.keys()))

# 快速測試
print("\n── 工具測試 ──")
print(web_search("GPT 最新消息"))
print(calculator("2^10 + sqrt(144)"))
print(get_current_time())

### 1-2 OpenAI Function Calling 的 JSON Schema

OpenAI Tool Calling API 需要我們以 JSON Schema 格式描述每個工具的簽名（名稱、描述、參數類型）。
這份 Schema 會傳給模型，讓它知道「有哪些工具可用、每個工具需要什麼參數」。

In [ ]:
# OpenAI tool_calls API 所需的工具定義（JSON Schema 格式）
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "搜索網路上的最新資訊。當需要了解近期新聞、產品發布或最新技術動態時使用。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索關鍵字或問題，使用繁體中文或英文",
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "計算數學表達式。支援加減乘除、次方（^）、sqrt、sin、cos、log。",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "數學表達式，例如 '2^10 + sqrt(144)' 或 '(100 * 1.05)^3'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "rag_search",
            "description": "從內部知識庫搜索關於 LLM、RAG、向量資料庫、雲端服務等技術文件。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索查詢，例如 'RAG 是什麼' 或 'ChromaDB 特色'",
                    },
                    "n_results": {
                        "type": "integer",
                        "description": "回傳的最大文件數量，預設 3",
                        "default": 3,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "取得當前的日期與時間。當問題涉及「現在幾點」或「今天日期」時使用。",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
]

print("✓ TOOLS_SCHEMA 定義完成，共", len(TOOLS_SCHEMA), "個工具")
print("\n工具名稱與描述：")
for t in TOOLS_SCHEMA:
    fn = t["function"]
    print(f"  • {fn['name']:20s} — {fn['description'][:50]}...")

---
## Part 2：單一 Tool Call 示範

### 2-1 原始 API 請求/回應流程

在正式實作 ReAct 之前，先了解 OpenAI Tool Calling API 的原始格式：

```
User 問題
    ↓
LLM 回傳 tool_calls（JSON）
    ↓
我們執行函數並取得結果
    ↓
把結果以 role="tool" 傳回 LLM
    ↓
LLM 生成最終回答
```

In [ ]:
def execute_tool_call(tool_call) -> str:
    """解析單一 tool_call 物件並執行對應函數。"""
    fn_name = tool_call.function.name
    fn_args = json.loads(tool_call.function.arguments)

    if fn_name not in TOOL_REGISTRY:
        return f"[錯誤] 未知工具：{fn_name}"

    fn = TOOL_REGISTRY[fn_name]
    try:
        result = fn(**fn_args)
        return result
    except Exception as exc:
        return f"[工具執行錯誤] {fn_name}: {exc}"


def simple_tool_loop(user_message: str, verbose: bool = True) -> str:
    """
    簡單的 tool call 迴圈：
    持續呼叫 LLM，直到沒有更多 tool_calls 為止。
    """
    messages = [
        {"role": "user", "content": user_message}
    ]

    if verbose:
        print(f"🙋 User: {user_message}\n")

    while True:
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
            tool_choice="auto",
        )
        msg = response.choices[0].message

        # 若模型沒有呼叫工具，直接回傳文字
        if not msg.tool_calls:
            if verbose:
                print(f"🤖 Assistant: {msg.content}")
            return msg.content

        # 處理每個 tool_call
        messages.append(msg)  # 把助理的訊息（含 tool_calls）加入歷史

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            if verbose:
                print(f"🔧 Tool Call: {fn_name}({fn_args})")

            result = execute_tool_call(tc)
            if verbose:
                print(f"📋 Result   : {result[:120]}..." if len(result) > 120 else f"📋 Result   : {result}")
                print()

            # 把工具結果加入對話歷史
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })


# ── 示範 ──────────────────────────────────────────────────────────
print("=" * 60)
print("示範 1：計算機工具")
print("=" * 60)
simple_tool_loop("請幫我計算 2 的 10 次方加上 144 的平方根")

In [ ]:
print("=" * 60)
print("示範 2：RAG 搜索工具")
print("=" * 60)
simple_tool_loop("ChromaDB 有什麼特色？與其他向量資料庫相比有何優勢？")

In [ ]:
print("=" * 60)
print("示範 3：多工具組合（時間 + 網路搜索）")
print("=" * 60)
simple_tool_loop("現在幾點？另外幫我搜索一下最新的 Claude 模型資訊")

### 2-2 觀察 tool_calls 的原始結構

讓我們直接印出 API 回傳的原始 `tool_calls` 物件，了解其資料結構。

In [ ]:
# 直接觀察原始 tool_calls 結構
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "計算 sqrt(256) 然後告訴我 RAG 是什麼"}],
    tools=TOOLS_SCHEMA,
    tool_choice="auto",
)

msg = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print(f"\ntool_calls 數量: {len(msg.tool_calls) if msg.tool_calls else 0}")

if msg.tool_calls:
    for i, tc in enumerate(msg.tool_calls):
        print(f"\n── tool_call [{i}] ──")
        print(f"  id        : {tc.id}")
        print(f"  type      : {tc.type}")
        print(f"  fn.name   : {tc.function.name}")
        print(f"  fn.args   : {tc.function.arguments}")

---
## Part 3：ReAct 迴圈實作

### ReAct 核心概念

ReAct 的全名是 **Re**asoning + **Act**ing，核心流程如下：

```
┌─────────────────────────────────────────────┐
│                  ReAct Loop                 │
│                                             │
│  Thought: 「我需要先搜索 X 再計算 Y...」     │
│      ↓                                      │
│  Action: 呼叫 rag_search("X")               │
│      ↓                                      │
│  Observation: 「知識庫回傳了以下文件...」    │
│      ↓                                      │
│  Thought: 「根據文件，我還需要...」           │
│      ↓                                      │
│  Action: 呼叫 calculator("...")             │
│      ↓                                      │
│  Observation: 「計算結果為...」              │
│      ↓                                      │
│  Final Answer: 「綜合以上資訊...」           │
└─────────────────────────────────────────────┘
```

關鍵差異：相比於 Part 2 的「簡單工具迴圈」，ReAct 的 System Prompt 明確要求模型**在呼叫工具前先輸出 Thought**，這使得推理過程更透明、可解釋。

In [ ]:
# ReAct System Prompt
REACT_SYSTEM_PROMPT = """\
你是一個能夠使用工具的 AI 助理，請嚴格遵循 ReAct 框架（Reasoning + Acting）。

在每一步驟，你必須按照以下格式思考：
  Thought: [分析當前情況，決定下一步要做什麼]
  Action: [決定呼叫哪個工具，或是已有足夠資訊可以回答]

當你呼叫工具後，系統會提供 Observation（工具執行結果），
然後你繼續進行下一輪的 Thought → Action。

當你認為已有足夠資訊回答使用者問題時，請直接給出最終答案，
不要再呼叫工具。最終回答請盡量詳細且有條理。

可用工具：
- web_search(query): 搜索網路最新資訊
- calculator(expression): 計算數學表達式
- rag_search(query): 搜索內部知識庫（LLM/RAG/雲端技術）
- get_current_time(): 取得當前時間
"""


def react_loop(
    user_question: str,
    max_iterations: int = 6,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    從零實作的 ReAct 迴圈。

    Returns:
        dict with keys: answer, steps, iterations_used
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user",   "content": user_question},
    ]

    steps = []  # 記錄每個步驟
    iteration = 0

    if verbose:
        print("=" * 65)
        print(f"🙋 問題：{user_question}")
        print("=" * 65)

    while iteration < max_iterations:
        iteration += 1

        if verbose:
            print(f"\n── 迭代 {iteration} ──────────────────────────")

        # ── 呼叫 LLM ──────────────────────────────────────────────
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # ── 若模型輸出文字思考（Thought）────────────────────────
        if msg.content:
            if verbose:
                print(f"💭 Thought / Answer:\n{msg.content}")
            steps.append({"type": "thought", "content": msg.content})

        # ── 若沒有工具呼叫 → 最終回答 ─────────────────────────
        if not msg.tool_calls:
            if verbose:
                print("\n✅ ReAct 迴圈結束（無更多 tool_calls）")
            return {
                "answer": msg.content,
                "steps": steps,
                "iterations_used": iteration,
            }

        # ── 執行工具 ────────────────────────────────────────────
        messages.append(msg)  # 加入助理訊息（含 tool_calls）

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)

            if verbose:
                print(f"\n🔧 Action: {fn_name}({fn_args})")

            observation = execute_tool_call(tc)

            if verbose:
                obs_preview = observation[:200] + "..." if len(observation) > 200 else observation
                print(f"👁  Observation:\n{obs_preview}")

            steps.append({
                "type": "action",
                "tool": fn_name,
                "args": fn_args,
                "observation": observation,
            })

            # 把觀察結果加入對話
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": observation,
            })

    # ── 超出最大迭代次數 ────────────────────────────────────────
    if verbose:
        print(f"\n⚠️  已達最大迭代次數 ({max_iterations})，強制停止")

    # 強制要求模型給出最終回答
    messages.append({"role": "user", "content": "請根據目前收集到的所有資訊，給出最終回答。"})
    final_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
    )
    final_answer = final_resp.choices[0].message.content

    if verbose:
        print(f"\n🤖 最終回答（強制）：\n{final_answer}")

    return {
        "answer": final_answer,
        "steps": steps,
        "iterations_used": max_iterations,
    }


print("✓ react_loop() 定義完成")

In [ ]:
# ── ReAct 測試 1：需要 RAG + 計算的複合問題 ──────────────────────
result1 = react_loop(
    "ReAct 框架是什麼？如果我有一個問題需要呼叫 3 次工具，"
    "每次工具呼叫需要 500ms，總共需要多少毫秒？"
)
print(f"\n📊 使用了 {result1['iterations_used']} 次迭代，共 {len(result1['steps'])} 個步驟")

In [ ]:
# ── ReAct 測試 2：需要網路搜索 + 知識庫的問題 ────────────────────
result2 = react_loop(
    "請比較 GPT-4o 和 Claude 3.5 的最新發展，並說明 RAG 如何幫助這些模型處理知識邊界問題"
)
print(f"\n📊 使用了 {result2['iterations_used']} 次迭代")

### 3-1 ReAct vs 單次查詢（Naive RAG）比較

| 面向 | Naive Single-Shot RAG | ReAct Agent |
|------|----------------------|-------------|
| **思考過程** | 不透明，一次生成 | 逐步可見（Thought → Action → Observation）|
| **工具使用** | 固定 1 次 RAG 檢索 | 動態決定，可呼叫多個不同工具 |
| **複雜問題** | 一次性回答，品質不穩定 | 迭代精煉，逐步補充資訊 |
| **計算能力** | 無（依賴模型內部估算）| 可呼叫 calculator 精確計算 |
| **即時資訊** | 無（受訓練截止日限制）| 可呼叫 web_search / get_current_time |
| **延遲** | 低（1 次 LLM call）| 較高（多次 LLM call + 工具執行）|
| **成本** | 低 | 較高（多次 API 呼叫）|
| **可解釋性** | 低 | 高（每步可審查）|

In [ ]:
# ── 直接比較：Naive RAG vs ReAct ─────────────────────────────────
test_question = "Tool Calling 在 LLM 架構中的角色是什麼？同時幫我算 1024 * 768"

print("=" * 65)
print("[Naive Single-Shot RAG]")
print("=" * 65)

# Naive：先手動 RAG，然後一次性詢問 LLM
naive_context = rag_search("Tool Calling LLM")
naive_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": f"請根據以下背景資料回答問題：\n\n{naive_context}"},
        {"role": "user", "content": test_question},
    ]
)
naive_answer = naive_response.choices[0].message.content
print(f"回答：\n{naive_answer}")

print("\n" + "=" * 65)
print("[ReAct Agent]")
print("=" * 65)
result3 = react_loop(test_question)

---
## Part 4：Self-Reflection（自我反思）

### 概念說明

ReAct 的最終答案可能仍有問題：
- **幻覺**：模型補充了知識庫中不存在的資訊
- **遺漏**：問題某些面向沒有被回答
- **矛盾**：不同工具的結果沒有被正確整合

**Self-Reflection** 流程：
```
ReAct 產生草稿答案
    ↓
「批判者」LLM 審查草稿（是否有根據？是否完整？有無幻覺？）
    ↓
若發現問題 → 補充檢索 + 修正
    ↓
最終優化答案
```

In [ ]:
CRITIC_SYSTEM_PROMPT = """\
你是一個嚴格的 AI 回答品質審核員。你的工作是批判性地分析另一個 AI 所給出的回答。

請從以下三個角度評估回答：
1. **事實依據**：回答中是否有未經工具結果支持的聲明（潛在幻覺）？
2. **完整性**：原始問題的每個部分都有被回答嗎？
3. **邏輯一致性**：各個工具的觀察結果有被正確整合嗎？

輸出格式：
評分：[1-10 分，10 分為完美]
問題摘要：[列出發現的問題，若無則寫「無明顯問題」]
補充建議：[建議要補充哪些資訊，或搜索哪些關鍵字，若無則寫「無需補充」]
"""


def self_reflect_and_improve(
    original_question: str,
    draft_answer: str,
    react_steps: List[Dict],
    score_threshold: int = 7,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    對 ReAct 草稿答案進行自我反思並改進。

    Args:
        original_question: 原始使用者問題
        draft_answer: ReAct 產生的草稿答案
        react_steps: ReAct 執行過程中的步驟記錄
        score_threshold: 評分低於此閾值才進行改進（1-10）

    Returns:
        dict with: score, critique, final_answer, improved
    """
    if verbose:
        print("\n" + "=" * 65)
        print("🪞 Self-Reflection 階段")
        print("=" * 65)

    # 整理工具執行結果供批判者參考
    observations_summary = []
    for step in react_steps:
        if step["type"] == "action":
            observations_summary.append(
                f"工具：{step['tool']}，參數：{step['args']}\n"
                f"結果：{step['observation'][:200]}"
            )

    critic_user_msg = f"""\
原始問題：{original_question}

工具執行記錄：
{chr(10).join(observations_summary) if observations_summary else '（無工具呼叫）'}

AI 草稿回答：
{draft_answer}

請評估這個回答的品質。"""

    # ── 呼叫批判者 LLM ────────────────────────────────────────────
    critic_response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": CRITIC_SYSTEM_PROMPT},
            {"role": "user",   "content": critic_user_msg},
        ]
    )
    critique = critic_response.choices[0].message.content

    if verbose:
        print(f"\n🔍 批判者評估：\n{critique}")

    # ── 解析評分 ──────────────────────────────────────────────────
    score_match = re.search(r'評分[：:]\s*(\d+)', critique)
    score = int(score_match.group(1)) if score_match else 5

    if verbose:
        print(f"\n📊 解析評分：{score}/10")

    # ── 若評分足夠高，直接回傳原始答案 ──────────────────────────
    if score >= score_threshold:
        if verbose:
            print(f"\n✅ 評分達標（{score} >= {score_threshold}），無需改進")
        return {
            "score": score,
            "critique": critique,
            "final_answer": draft_answer,
            "improved": False,
        }

    # ── 評分不足，嘗試補充一輪 RAG 並改進 ────────────────────────
    if verbose:
        print(f"\n⚡ 評分不足（{score} < {score_threshold}），進行補充改進...")

    # 從批判意見中提取補充建議的關鍵字
    supplement_context = rag_search(original_question, n_results=2)
    web_supplement = web_search(original_question[:30])  # 用問題前30字搜索

    improvement_prompt = f"""\
你之前回答了以下問題，但審核員發現了一些問題：

原始問題：{original_question}

你的草稿回答：
{draft_answer}

審核員的批評：
{critique}

補充資料（知識庫）：
{supplement_context}

補充資料（網路）：
{web_supplement}

請根據審核員的批評與補充資料，改進你的回答，確保準確完整。"""

    improved_response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": improvement_prompt}]
    )
    improved_answer = improved_response.choices[0].message.content

    if verbose:
        print(f"\n🔄 改進後回答：\n{improved_answer}")

    return {
        "score": score,
        "critique": critique,
        "final_answer": improved_answer,
        "improved": True,
    }


print("✓ self_reflect_and_improve() 定義完成")

In [ ]:
# ── 完整流程示範：ReAct + Self-Reflection ────────────────────────
question = "Agent 架構與傳統 LLM 有什麼不同？向量資料庫在其中扮演什麼角色？"

print("Phase 1：執行 ReAct 迴圈...\n")
react_result = react_loop(question, max_iterations=4)
draft = react_result["answer"]

print("\n" + "-" * 65)
print("Phase 2：執行 Self-Reflection...")

reflection_result = self_reflect_and_improve(
    original_question=question,
    draft_answer=draft,
    react_steps=react_result["steps"],
)

In [ ]:
# ── Before vs After 比較 ──────────────────────────────────────────
print("=" * 65)
print("BEFORE（ReAct 草稿回答）")
print("=" * 65)
print(draft)

print("\n" + "=" * 65)
if reflection_result["improved"]:
    print("AFTER（Self-Reflection 改進後）")
else:
    print("AFTER（評分達標，無需改進）")
print("=" * 65)
print(reflection_result["final_answer"])

print(f"\n📊 最終評分：{reflection_result['score']}/10")
print(f"📊 是否改進：{'是' if reflection_result['improved'] else '否'}")

### 4-1 完整 ReAct + Self-Reflection Pipeline 封裝

In [ ]:
def full_react_pipeline(
    question: str,
    max_iterations: int = 6,
    score_threshold: int = 7,
    verbose: bool = True,
) -> str:
    """
    完整的 ReAct + Self-Reflection Pipeline。
    回傳最終的高品質回答。
    """
    # Step 1: ReAct 迴圈
    react_result = react_loop(question, max_iterations=max_iterations, verbose=verbose)

    # Step 2: Self-Reflection
    reflection = self_reflect_and_improve(
        original_question=question,
        draft_answer=react_result["answer"],
        react_steps=react_result["steps"],
        score_threshold=score_threshold,
        verbose=verbose,
    )

    if verbose:
        print("\n" + "=" * 65)
        print("🏁 最終回答")
        print("=" * 65)
        print(reflection["final_answer"])
        print(f"\n（ReAct 迭代：{react_result['iterations_used']} 次｜"
              f"評分：{reflection['score']}/10｜"
              f"是否改進：{'是' if reflection['improved'] else '否'}）")

    return reflection["final_answer"]


# 最終展示
final_answer = full_react_pipeline(
    "現在幾點？另外請解釋 Self-RAG 是什麼，並計算如果每天節省 15% 的 API 費用，"
    "一個月（30天）後總共省了幾個百分比的費用？"
)

---
## Part 5：面試 Q&A

以下整理了 Google FDE（Field Deployment Engineer / Forward Deployed Engineer）面試中常見的 ReAct 與 Tool Calling 相關題目。

### Q1：什麼是 ReAct 框架？它如何改進傳統 LLM 回答複雜問題的能力？

**A：**

ReAct（Reasoning + Acting）是 Yao et al. 2022 提出的框架，核心思想是讓 LLM 在「思考（Thought）」和「行動（Action）」之間交替進行：

1. **Thought（思考）**：模型內部推理，決定下一步要做什麼
2. **Action（行動）**：呼叫外部工具（搜索、計算機、資料庫查詢…）
3. **Observation（觀察）**：取得工具執行結果，作為下一輪思考的輸入

相比單次 RAG（先檢索，再生成），ReAct 的優勢在於：
- **動態多步推理**：可依序呼叫多個工具，每步都能根據新資訊調整策略
- **可解釋性**：Thought 步驟明確記錄推理過程，便於 Debug
- **工具多樣性**：不限於 RAG，可整合計算機、API、資料庫等任意工具
- **錯誤恢復**：若某個工具返回錯誤，Thought 可識別並嘗試替代方案

**實際面試回答技巧**：可以與 Chain-of-Thought（CoT）對比，CoT 是純粹的思考鏈，而 ReAct = CoT + 工具呼叫能力。

### Q2：OpenAI Tool Calling API 的工作原理是什麼？從 API 層面說明整個流程。

**A：**

OpenAI Tool Calling 的流程分為四個步驟：

**步驟 1：定義工具 Schema**
```json
{
  "type": "function",
  "function": {
    "name": "web_search",
    "description": "搜索網路資訊",
    "parameters": {
      "type": "object",
      "properties": { "query": {"type": "string"} },
      "required": ["query"]
    }
  }
}
```

**步驟 2：發送請求（包含 tools 和 tool_choice）**
- `tool_choice: "auto"` — 讓模型自行決定是否呼叫工具
- `tool_choice: "required"` — 強制必須呼叫工具
- `tool_choice: {"type": "function", "function": {"name": "X"}}` — 指定呼叫特定工具

**步驟 3：解析回應中的 `tool_calls`**
- `finish_reason` 為 `"tool_calls"` 時代表模型想呼叫工具
- 每個 tool_call 包含：`id`、`function.name`、`function.arguments`（JSON 字串）

**步驟 4：回傳工具結果**
```python
{"role": "tool", "tool_call_id": tc.id, "content": result}
```

**關鍵點**：模型本身不執行任何工具，它只是「宣告意圖」，實際執行由開發者負責。這個設計保持了 LLM 的無狀態性，同時讓工具執行環境完全可控。

### Q3：如何設計一個具備停止條件和錯誤處理的 ReAct Agent？說明你的設計考量。

**A：**

一個生產級 ReAct Agent 需要以下停止條件和錯誤處理策略：

**停止條件：**
1. **正常停止**：`finish_reason != "tool_calls"`，模型自行判斷資訊足夠
2. **迭代上限**：設定 `max_iterations=6`（避免無限迴圈），超出時強制總結
3. **時間限制**：設定全局超時（例如 30 秒），適用於生產環境
4. **Token 預算**：監控 token 累計使用量，超限時停止

**錯誤處理：**
```python
try:
    result = tool_fn(**args)
except Exception as e:
    result = f"[工具執行錯誤] {e}"  # 錯誤訊息也作為 Observation 回傳
    # 讓模型自行決定是否重試或換用其他工具
```

**關鍵設計原則：**
- **工具失敗不中斷**：錯誤訊息作為 Observation 回傳，讓 LLM 自行決策
- **冪等性**：工具應設計為可重複呼叫不產生副作用（對讀取操作天然成立）
- **白名單 eval**：計算工具絕對不能用原生 `eval`，需限制命名空間
- **日誌記錄**：每個 Action/Observation 都應記錄，方便事後 Debug 和審計

**面試加分點**：可以提到 Exponential Backoff 重試機制、Circuit Breaker 模式，以及如何整合 OpenTelemetry 追蹤每次工具呼叫的延遲。

### Q4：Self-Reflection 在 LLM Agent 中的作用是什麼？與強化學習中的自我評估有何類比？

**A：**

**Self-Reflection 的核心作用：**

LLM 產生的回答可能存在幻覺（hallucination）、遺漏或邏輯錯誤。Self-Reflection 透過引入「批判者」（Critic）角色，在回傳給使用者前對回答進行品質把關：

1. **幻覺偵測**：檢查回答中是否有工具觀察結果無法支持的聲明
2. **完整性檢查**：確保問題的每個面向都被回答
3. **一致性驗證**：多個工具的結果是否被正確整合

**與強化學習的類比：**

| 強化學習概念 | Self-Reflection 類比 |
|-------------|---------------------|
| Policy（策略）| ReAct Agent（生成答案）|
| Reward Function（獎勵函數）| Critic LLM（評分）|
| Value Estimation | 評分數值（1-10 分）|
| Policy Improvement | 根據批評改進回答 |
| RLHF | 用人類回饋訓練更好的 Critic |

**相關論文參考：**
- **Reflexion**（Shinn et al. 2023）：讓 Agent 從失敗經驗中學習，將反思結果存入「episodic memory」
- **Self-RAG**（Asai et al. 2023）：在訓練階段加入反思 token，讓模型學會「何時需要檢索」
- **Constitutional AI**（Anthropic）：讓模型用一組原則批判自己的回答

**實際應用考量**：Self-Reflection 會增加額外的 LLM 呼叫成本（約 20-50%），需要在回答品質與成本之間取得平衡，可設定評分閾值（如 ≥7 分不觸發改進）來控制成本。

---
## 本章小結

本章從零實作了完整的 ReAct + Tool Calling + Self-Reflection Pipeline：

| 主題 | 核心概念 | 關鍵 API/函數 |
|------|---------|---------------|
| Tool Calling | LLM 宣告工具呼叫意圖，開發者執行 | `tool_calls`、`role: "tool"` |
| Tool Registry | 工具名稱 → Python 函數的映射表 | `dict[str, Callable]` |
| ReAct 迴圈 | Thought → Action → Observation 循環 | `react_loop()` |
| 停止條件 | `max_iterations` + 無 tool_calls 時停止 | `finish_reason` |
| Self-Reflection | Critic LLM 批判草稿，低分時補充改進 | `self_reflect_and_improve()` |

**下一步建議：**
- 嘗試實作 **Parallel Tool Calling**（多個工具並行執行）
- 加入 **Memory**（把過去對話與反思結果存入向量庫）
- 實作 **Planning**（先規劃步驟，再依序執行）
- 探索 LangGraph / LlamaIndex 等框架如何將以上概念產品化